In [1]:
import numpy as np

## Test Input.py

In [2]:
from fsvisual.input import read_energy_numbers

df, fermi_energy, rez_base_vect, grid_size=read_energy_numbers("bxsf/W_bcc_161x161x161")

np.savez(
    "npz/expected_W_bcc_161x161x161.npz",
    df=df,
    fermi_energy=fermi_energy,
    rez_base_vect=rez_base_vect,
    grid_size=grid_size
)

## Test brillouin_zone.py

In [3]:
basis1 = [[0.8341935050, 0.8341935050, -0.8341935050],
          [-0.8341935050, 0.8341935050, 0.8341935050],
          [0.8341935050, -0.8341935050, 0.8341935050]]

basis2 = [[0.6231489608, 0.6231489608, -0.6231489608],
          [-0.6231489608, 0.6231489608, 0.6231489608],
          [0.6231489608, -0.6231489608, 0.6231489608]]

basis3 = [[1.140682526, 0.6585733636, 0.000000000],
          [0.000000000, 1.317146727, 0.000000000],
          [0.000000000, 0.000000000, 0.6106219722]]

basis4 = [[0.000000000, 1.043564131, 1.043564131],
          [1.043564131, 0.000000000, 1.043564131],
          [1.043564131, 1.043564131, 0.000000000]]


np.savez(
    "brillouin_zone/reciprocal_basis_examples.npz",
    basis1=basis1,
    basis2=basis2,
    basis3=basis3,
    basis4=basis4
)

In [32]:
from fsvisual.brillouin_zone import first_bz

bz_output = [first_bz(basis1), first_bz(basis2), first_bz(basis3), first_bz(basis4)]

for i in range(len(bz_output)):
    for j in range(len(bz_output[i][0])):
        for k in range(len(bz_output[i][0][j])):
            if bz_output[i][0][j][k] is None:
                bz_output[i][0][j][k] = np.nan

np.savez(
    "brillouin_zone/bz_output_xyz.npz",
    output1=bz_output[0][0],
    output2=bz_output[1][0],
    output3=bz_output[2][0],
    output4=bz_output[3][0]
)

i = [len(array[0][1]) for array in bz_output]
my_dict = {"output1": bz_output[0][1][:i[0]],
           "output2": bz_output[1][1][:i[1]],
           "output3": bz_output[2][1][:i[2]],
           "output4": bz_output[3][1][:i[3]]}

for j in range(len(bz_output)):
    np.savez(
        f"brillouin_zone/bz_output/output{j}.npz",
        *bz_output[j][1][:i[j]]
    )



## Test mesh_algorythm.py

In [9]:
# create cartesian mesh
from fsvisual.mesh_algorithms import create_cartesian_mesh

grid_size1 = [40, 40, 40]
grid_size2 = [30, 30, 10]
grid_size3 = [10, 20, 30]

my_mesh1 = create_cartesian_mesh(grid_size1)
my_mesh2 = create_cartesian_mesh(grid_size2)
my_mesh3 = create_cartesian_mesh(grid_size3)

np.savez("mesh_algorithms/cartesian_meshes", mesh1=my_mesh1, mesh2 = my_mesh2, mesh3 = my_mesh3)



In [40]:
# triangulate faces
from fsvisual.mesh_algorithms import triangulate_faces
import copy
tf_bz_output = copy.deepcopy(bz_output[0][1])
tf_bz_output.append([[0,1,1], [0,0,2], [0,3,3]])
tf_bz_output.append([[0,1,1], [0,0,2], [0,3,3], [5,4,3]])
np.savez("mesh_algorithms/output_triangulate_faces.npz", faces=triangulate_faces(tf_bz_output))

In [29]:
from fsvisual.mesh_algorithms import triangle_area

test_triangle = [[2.5, 10.5, 4.7], [10, 1.6, -20], [-10.2, 8.9, 3]]
print(triangle_area(test_triangle))

175.207339030076


In [13]:
from fsvisual.mesh_algorithms import triangle_center

test_triangle = [[2.5, 55.2, 4.7], [12, 1.6, -20], [-10.2, 8.9, 3]]

print(triangle_center(test_triangle))

[1.4333333333333336, 21.9, -4.1]


In [33]:
from fsvisual.mesh_algorithms import face_center_BZ
expected_face_center_BZ = face_center_BZ(bz_output[0][1])

np.savez("mesh_algorithms/output_face_centers_bz", face_centers=expected_face_center_BZ)

## Test visualisation.py

In [41]:
from fsvisual.visualisation import  build_plotly_figure
from fsvisual.fermisurface import FermiSurface

my_surface = FermiSurface()
my_surface.build_surface_with_bxsf_files("bxsf/fcc_41x41x41")

In [ ]:
my_surface.fermi_surface_list[0].export("mesh.ply")

In [6]:
print(my_surface.band_index)

NameError: name 'my_surface' is not defined

In [60]:
import json

fig = build_plotly_figure(my_surface.fermi_surface_list, my_surface.brillouin_zone, my_surface.band_index)

fig_json = fig.to_json()
with open("visualisation/figure.json", "w") as f:
    json.dump(fig_json, f)


In [ ]:
import plotly.io as io

with open("visualisation/figure.json", "r") as f:
    fig_json = json.load(f)
loaded_fig = io.from_json(fig_json)

# Zeige die geladene Figur an
loaded_fig.show()


In [14]:
from fsvisual.visualisation import build_plotly_figure
from fsvisual.brillouin_zone import first_bz
import trimesh
import json
import plotly.io as io


#surface_list = trimesh.load("tests/data/visualisation/mesh.ply")    # mesh from bxsf/fcc_41x41x41
#rez_lattice = [[0.8152569492, 0.8152569492, -0.8152569492], #[0.8152569492, -0.8152569492, 0.8152569492],
 #              [-0.8152569492, 0.8152569492 , 0.8152569492]]
#brillouin_zone_obj = first_bz(rez_lattice)

URLError: <urlopen error [Errno 11001] getaddrinfo failed>